# Nonlinear MPC swing-up of the double pendulum with `acados` — Student Version

In this exercise you will implement a **Nonlinear Model-Predictive Controller (NMPC)**
that swings up the real CloudPendulum double pendulum, using [`acados`](https://docs.acados.org).

[**acados**](https://docs.acados.org) is an open-source software package for
the efficient solution of optimal control and estimation problems. It is
written in C and designed for embedded / real-time applications, so it is a
popular state-of-the art choice when you want to run nonlinear MPC at kHz rates on real
hardware. 
It is actively developed by the group of Moritz Diehl from the University of Freiburg, Germany.
In this exercise we interact with acados through its Python
interface ([`acados_template`](https://docs.acados.org/python_interface/index.html)),
which builds the model, generates efficient C code for the specific problem,
compiles it, and hands us back a Python object we can call in the control
loop. A typical workflow is:

1. Describe the continuous-time dynamics symbolically as an `AcadosModel`.
2. Formulate the OCP (horizon, cost, constraints, solver options) as an
   `AcadosOcp`.
3. Let acados generate and compile a problem-specific solver
   (`AcadosOcpSolver`).
4. Query the solver in the closed-loop controller.

The symbolic modelling is done with [**CasADi**](https://web.casadi.org/),
a framework for algorithmic differentiation and numerical optimization.
CasADi lets us write equations of motion with ordinary-looking Python code
(`ca.sin`, `ca.vertcat`, matrix algebra, `@`, …), and then hands the
resulting symbolic expressions over to acados for code generation. If you
need a refresher, the CasADi [documentation](https://web.casadi.org/docs/)
is very readable.

## What you will do

You will work through five tasks:

1. **Task 1 – acados model**  
   Derive the continuous-time dynamics of the double pendulum in CasADi and
   wrap them in an `AcadosModel`.
2. **Task 2 – OCP definition**  
   Formulate the optimal control problem (cost, constraints, solver options)
   as an `AcadosOcp`.
3. **Task 3 – Solver generation**  
   Create the `AcadosOcpSolver` object, which in the background compiles efficient C-code for us.
4. **Task 4 – Closed-loop controller**  
   Implement `run_controller(...)` that measures the state, calls the
   solver and sends the torques to the real robot, then plots everything.
5. **Task 5 – Experiments**  
   Try the fully-actuated pendulum, then attempt the same swing-up for the
   **pendubot** and **acrobot** (only one joint actuated).

The state and input vectors you must use throughout the whole exercise are

$$
x = \begin{bmatrix} q_1 \\ q_2 \\ \dot q_1 \\ \dot q_2 \end{bmatrix} \in \mathbb{R}^4,
\qquad
u = \begin{bmatrix} u_1 \\ u_2 \end{bmatrix} \in \mathbb{R}^2.
$$


## 0. Setup

Install `acados` (one-time; skip if it has already been built in your workspace)
and import everything we need.

In [ ]:
from helper import install_acados
install_acados()

In [ ]:
import os
import sys
import ctypes
import time
import shutil
import numpy as np
import casadi as ca
import matplotlib.pyplot as plt
from ctypes import RTLD_GLOBAL
from IPython.display import Video, display
from cloudpendulumclient.client import Client
from acados_template import AcadosModel, AcadosOcp, AcadosOcpSolver

from helper import (
    download_video, plot_run, show_video,
    run_controller_simulation, animate_swingup_simulation,
)

# --- acados library paths (do not change) ---
acados_dir = os.path.expanduser("~/acados")
lib_dir = os.path.join(acados_dir, "lib")
os.environ["ACADOS_SOURCE_DIR"] = acados_dir
os.environ["LD_LIBRARY_PATH"] = f"{acados_dir}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
if f"{acados_dir}/interfaces/acados_template" not in sys.path:
    sys.path.insert(0, f"{acados_dir}/interfaces/acados_template")

print("Using lib dir:", lib_dir)
print("Exists:", os.path.exists(lib_dir))
print("Contents:", os.listdir(lib_dir)[:20])
# make python package visible
sys.path.insert(0, os.path.join(acados_dir, "interfaces", "acados_template"))
# preload shared libraries explicitly
for lib in [
    "libblasfeo.so",
    "libhpipm.so",
    "libacados.so",
]:
    path = os.path.join(lib_dir, lib)
    if os.path.exists(path):
        print("Loading", path)
        ctypes.CDLL(path, mode=RTLD_GLOBAL)
    else:
        print("Not found:", path)
os.environ["ACADOS_SOURCE_DIR"] = acados_dir
from acados_template import AcadosOcp, AcadosOcpSolver
print("acados_template import works")

# --- clean up old generated code so we never run stale C code ---
code_export_dir = "./c_generated_code"
if os.path.exists(code_export_dir):
    shutil.rmtree(code_export_dir)

## 1. Physical parameters of the CloudPendulum (given)

These are the identified parameters of the real robot.  
**Change them if you have new parameters from previous exercises**

In [ ]:
# --- identified parameters of the CloudPendulum double pendulum ---
I1  = 0.00046166221821039165   # link-1 inertia about its own CoM axis    [kg m^2]
I2  = 0.00023702395072092597   # link-2 inertia                            [kg m^2]
Ir  = 6.1e-06                  # rotor inertia                             [kg m^2]
b1  = 7.634058385430087e-12    # viscous friction, joint 1
b2  = 0.0005106535523065844    # viscous friction, joint 2
cf1 = 0.0022933128359236008    # Coulomb friction, joint 1
cf2 = 0.0013188243178924417    # Coulomb friction, joint 2
g   = 9.81                     # gravity                                   [m/s^2]
gr  = 1.0                      # gear ratio
l1  = 0.05                     # length of link 1                          [m]
l2  = 0.05                     # length of link 2                          [m]
m1  = 0.10548177618443695      # mass of link 1                            [kg]
m2  = 0.07619744360415454      # mass of link 2                            [kg]
r1  = 0.05                     # CoM distance of link 1                    [m]
r2  = 0.03670036749567022      # CoM distance of link 2                    [m]
tl1 = 0.15                     # hardware torque limit joint 1             [Nm]
tl2 = 0.15                     # hardware torque limit joint 2             [Nm]

## Task 1 — Build the acados model

Write a function `build_model()` that returns an `AcadosModel` describing
the continuous-time dynamics

$$
M(q)\,\ddot q + C(q,\dot q)\,\dot q + F(\dot q) = B\,u + \tau_g(q),
$$

with

- **Mass matrix**
$$
M(q) = \begin{bmatrix}
I_1 + I_2 + m_2 l_1^2 + 2 m_2 l_1 r_2 \cos(q_2) + g_r^2 I_r + I_r &  I_2 + m_2 l_1 r_2 \cos(q_2) - g_r I_r \\
I_2 + m_2 l_1 r_2 \cos(q_2) - g_r I_r                          &  I_2 + g_r^2 I_r
\end{bmatrix}.
$$

- **Coriolis matrix**
$$
C(q,\dot q) = \begin{bmatrix}
-2 m_2 l_1 r_2 \sin(q_2)\dot q_2 & -m_2 l_1 r_2 \sin(q_2)\dot q_2 \\
\phantom{-} m_2 l_1 r_2 \sin(q_2)\dot q_1 & 0
\end{bmatrix}.
$$

- **Gravity torque**
$$
\tau_g(q) = \begin{bmatrix}
-m_1 g r_1 \sin q_1 - m_2 g\!\left(l_1 \sin q_1 + r_2 \sin(q_1+q_2)\right) \\
-m_2 g r_2 \sin(q_1+q_2)
\end{bmatrix}.
$$

- **Friction** (smooth Coulomb + viscous, `steepness = 100`):
$$
F(\dot q) = \begin{bmatrix}
b_1 \dot q_1 + c_{f1}\,\mathrm{atan}(100\,\dot q_1) \\
b_2 \dot q_2 + c_{f2}\,\mathrm{atan}(100\,\dot q_2)
\end{bmatrix}.
$$

- **Input matrix** `B = I_2` (fully actuated by default; this will be used
  to model pendubot / acrobot later by restricting torque limits).

> **What you have to do**
> 1. Create the CasADi symbols for `q1, q2, dq1, dq2` and build the state
>    vector `x = [q1, q2, dq1, dq2]`.
> 2. Create `u1, u2` and the input vector `u = [u1, u2]`.
> 3. Build `M, C, B`, the gravity `tau_g`, and the friction `F`.
> 4. Compute `qdd = M^{-1} (B u - C qd + tau_g - F)` and assemble both
>    the **explicit** and the **implicit** right-hand side:
>    - `f_expl = [qd; qdd]` — i.e. `xdot = f_expl(x, u)`.
>    - `f_impl = xdot - f_expl` — i.e. the residual form
>      `f_impl(xdot, x, u) = 0`.
>    
>    acados can use either formulation depending on the integrator / solver
>    you pick, so please fill in both `model.f_expl_expr` and
>    `model.f_impl_expr` — we might want to use both.
> 5. Fill in the remaining `AcadosModel` fields (`name`, `x`, `u`,
>    `xdot`) and return it.

Hints:
- Use `ca.SX.sym("name")` for scalar symbols and `ca.vertcat(...)` to stack.
- `ca.blockcat(a11, a12, a21, a22)` is convenient for 2×2 matrices.
- Invert the 2×2 matrix symbolically with `ca.solve(M, ca.SX.eye(2))`.

In [ ]:
def build_model():
    """Return an AcadosModel for the CloudPendulum double pendulum.

    State:  x = [q1, q2, dq1, dq2]
    Input:  u = [u1, u2]
    """
    # =====================================================================
    # TODO 1 — Build the symbolic double-pendulum model.
    #
    # Fill in the steps below:
    #   1. Create q1, q2, dq1, dq2, x, xdot, q, qd.
    #   2. Create u1, u2 and u.
    #   3. Build M(q), C(q, dq), B, tau_g(q), and F(dq).
    #   4. Compute qdd = M^{-1} (B u - C qd + tau_g - F).
    #   5. Assemble f_expl = [qd; qdd] and f_impl = xdot - f_expl.
    # =====================================================================

    # Type here: implement the CasADi symbols, dynamics, f_expl, and f_impl.
    raise NotImplementedError("TODO 1: implement the acados double-pendulum model.")

    # --- Wrap in an AcadosModel ---
    model = AcadosModel()
    model.name        = "double_pendulum"
    model.x           = x
    model.u           = u
    model.xdot        = xdot
    model.f_expl_expr = f_expl
    model.f_impl_expr = f_impl
    return model


## Task 2 — Build the optimal control problem

Write a function

```python
get_ocp(model, pred_horizon, mpc_dt, control_dt, weights,
        v_max, u_max_1, u_max_2)
```

that returns a fully configured `AcadosOcp`. The **solver options**
(QP solver, integrator, globalization, …) are already filled in for you —
they are not easy to find by trial and error. You only need to set up:

1. The prediction horizon (`N_horizon`, `tf`).
2. A `NONLINEAR_LS` cost. For swing-up we do **not** penalize raw angles
   directly. Instead we use a periodic representation

   $$
   y = [\sin q_1,\ \cos q_1,\ \sin q_2,\ \cos q_2,
        \dot q_1,\ \dot q_2,\ u_1,\ u_2].
   $$

   The terminal output is

   $$
   y_e = [\sin q_1,\ \cos q_1,\ \sin q_2,\ \cos q_2,
          \dot q_1,\ \dot q_2].
   $$

   This is important for swing-up: `q1 = pi`, `q1 = -pi`, and
   `q1 = 3*pi` are the same physical upright configuration, but a raw
   squared angle error would treat them as very different. The sin/cos
   representation lets the optimizer swing up through either direction.

   The upright reference `[q1, q2, dq1, dq2] = [pi, 0, 0, 0]` becomes

   $$
   y_\mathrm{ref} = [0, -1, 0, 1, 0, 0, 0, 0],
   \qquad
   y_{e,\mathrm{ref}} = [0, -1, 0, 1, 0, 0].
   $$

3. Box constraints on the inputs (`±u_max_1, ±u_max_2`) and on the joint
   velocities (`±v_max`).
4. An initial-state constraint (will be overwritten every MPC step).

The `weights` argument is a dictionary with five entries:

```python
weights = {
    "q":    [w_q1, w_q2],        # stage cost on each joint angle embedding
    "dq":   [w_dq1, w_dq2],      # stage cost on joint velocities
    "u":    [w_u1, w_u2],        # stage cost on torques
    "q_e":  [w_q1_e, w_q2_e],    # terminal cost on each joint angle embedding
    "dq_e": [w_dq1_e, w_dq2_e],  # terminal cost on joint velocities
}
```

Each angle weight is used twice: once for `sin(q)` and once for `cos(q)`.


In [ ]:
def get_ocp(model, pred_horizon, mpc_dt, control_dt, weights,
            v_max, u_max_1, u_max_2):
    """Build the acados OCP for the double pendulum swing-up."""
    ocp = AcadosOcp()
    ocp.model = model

    N  = pred_horizon
    Tf = pred_horizon * mpc_dt

    # --- 1) Horizon ---------------------------------------------------------
    # TODO 2a: set ocp.solver_options.N_horizon and ocp.solver_options.tf.

    # --- 2) Cost (NONLINEAR_LS with periodic angle embedding) --------------
    q1, q2, dq1, dq2 = ca.vertsplit(model.x)
    u1, u2           = ca.vertsplit(model.u)

    # TODO 2b: configure the NONLINEAR_LS stage and terminal costs.
    # Use the periodic angle embedding:
    #   y   = [sin(q1), cos(q1), sin(q2), cos(q2), dq1, dq2, u1, u2]
    #   y_e = [sin(q1), cos(q1), sin(q2), cos(q2), dq1, dq2]
    # The upright target [pi, 0, 0, 0] becomes:
    #   yref   = [0, -1, 0, 1, 0, 0, 0, 0]
    #   yref_e = [0, -1, 0, 1, 0, 0]
    # Build W and W_e by using each q/q_e weight twice, once for sin and
    # once for cos, then appending the dq and u weights.

    # --- 3) Constraints -----------------------------------------------------
    # TODO 2c: add box constraints for inputs and joint velocities, and set x0.

    # Keep this error until TODO 2a, TODO 2b, and TODO 2c are complete.
    raise NotImplementedError("TODO 2: configure the OCP horizon, cost, and constraints.")

    # --- 4) Solver options (given — keep as is) ----------------------------
    ocp.solver_options.qp_solver        = "PARTIAL_CONDENSING_HPIPM"
    ocp.solver_options.hpipm_mode       = "ROBUST"
    ocp.solver_options.hessian_approx   = "GAUSS_NEWTON"
    ocp.solver_options.integrator_type  = "IRK"
    ocp.solver_options.nlp_solver_type  = "SQP"
    ocp.solver_options.globalization    = "MERIT_BACKTRACKING"
    ocp.solver_options.with_adaptive_levenberg_marquardt = True
    ocp.solver_options.timeout_max_time = control_dt
    ocp.solver_options.qp_tol = 0.01
    ocp.solver_options.qp_solver_warm_start = 1
    ocp.solver_options.qp_solver_cond_N = int(N / 2)
    ocp.solver_options.print_level      = 0

    return ocp


## Task 3 — Generate the acados solver

Write `get_solver(ocp)` that calls `AcadosOcpSolver(...)` and returns the
compiled solver.

**What actually happens when you call `AcadosOcpSolver(ocp)`?**
Under the hood, acados takes your symbolic `AcadosModel` and `AcadosOcp`
description, generates problem-specific **C code** for the model, the
cost, the constraints and the chosen integrator, compiles it into a shared
library and then exposes it to Python through a thin ctypes wrapper.
In this notebook we interact with that compiled C solver through the
Python `AcadosOcpSolver` object — all the heavy lifting (derivatives,
QP solves, integration) is done in C for speed.

**Jupyter caching trick.** This C-code generation is also the reason for
the small trick below. The generated files are tagged by the `json_file`
name, and once a shared library is loaded into the running Python process
the Jupyter kernel will **keep using the already loaded `.so`**, even if
you change the model and re-run the cell. The most reliable fix is to
give each build a fresh `json_file` name (and therefore fresh generated
C files / shared library) using the current timestamp:

```python
build_tag = int(time.time())
json_file = f"double_pendulum_ocp_{build_tag}.json"
```

If your solver ever starts behaving as if your latest model changes are
being ignored, restarting the kernel is a safe escape hatch.

In [ ]:
def get_solver(ocp):
    """Build a fresh AcadosOcpSolver.

    Use a timestamped json_file to avoid stale compiled code between
    Jupyter reruns. Pass `verbose=False` so the compilation log stays short.
    """
    # =====================================================================
    # TODO 3 — Generate and return the acados solver.
    #
    # Hint:
    #   build_tag = int(time.time())
    #   json_file = f"double_pendulum_ocp_{build_tag}.json"
    #   solver = AcadosOcpSolver(ocp, json_file=json_file, verbose=False)
    # =====================================================================

    # Type here: create and return the solver.
    raise NotImplementedError("TODO 3: generate the AcadosOcpSolver.")


## Task 4 — Closed-loop controller

Implement

```python
run_controller(solver, u_max_1, u_max_2, v_max,
               Tf_exp=5.0, user_token=..., experiment_type="DoublePendulum")
```

that

1. Starts a CloudPendulum experiment of duration `Tf_exp`.
2. At every control tick:
   - reads the joint positions / velocities / torques from the robot,
   - builds the current state `x0 = [q1, q2, dq1, dq2]`,
   - fixes the initial state of the OCP via `solver.set(0, "lbx"/"ubx", x0)`,
   - calls `solver.solve()`,
   - reads the first optimal input `solver.get(0, "u")`,
   - clips it to the torque limits and sends it with `client.set_torque(...)`.
3. Logs `ts, xs, us, taus_meas, solver_status`.
4. Stops the experiment, downloads the video, and displays plots and video.

The real-time period `control_dt` is implicitly given by the solver
timeout you set in Task 2, so there is **no extra `sleep()`** needed: the
acados solver will not take longer than `control_dt`.

In [ ]:
def run_controller(solver,
                   u_max_1, u_max_2, v_max,
                   Tf_exp=5.0,
                   user_token="812115381621179648",
                   experiment_type="DoublePendulum"):
    """Run the NMPC closed loop on the real CloudPendulum, plot and show video."""

    # =====================================================================
    # TODO 4 — Implement the real-time NMPC loop.
    #
    # Your completed function should:
    #   1. Warm-start the SQP solver from x0 = [0, 0, 0, 0].
    #   2. Start a CloudPendulum experiment.
    #   3. Repeatedly read q, dq, and measured torques.
    #   4. Build x0 = [q1, q2, dq1, dq2] and clip dq to [-v_max, v_max].
    #   5. Set stage-0 lbx/ubx to x0.
    #   6. Solve the OCP and store the solver status.
    #   7. Read solver.get(0, "u"), clip to the torque limits, and send it.
    #   8. Log ts, xs, us, taus_meas, solver_status.
    #   9. Stop the experiment, download the video, then plot and show it.
    #
    # Keep this error until the whole controller is implemented. It is placed
    # before any hardware call so an incomplete notebook cannot start the robot.
    # =====================================================================
    raise NotImplementedError("TODO 4: implement the closed-loop NMPC controller.")

    # Suggested skeleton:
    #
    # x0_init = np.array([0.0, 0.0, 0.0, 0.0])
    # for i in range(2000):
    #     solver.set(0, "lbx", x0_init)
    #     solver.set(0, "ubx", x0_init)
    #     status = solver.solve()
    #     x0_init = solver.get(1, "x")
    #
    # client = Client()
    # session_token, livestream_url = client.start_experiment(...)
    # try:
    #     while (time.time() - start_loop) < Tf_exp:
    #         ...
    # finally:
    #     video_url = client.stop_experiment(session_token)


## 4.5 — Try the controller in simulation

Before running the underactuated cases on the real CloudPendulum, test the
same NMPC loop in simulation. The helper uses the same `build_model()`
dynamics as the OCP and returns the same kind of log dictionary as
`run_controller(...)`.


## Task 5 — Experiments

Now it is time to make the pendulum swing up. For each main experiment, use
the same workflow:

1. Build the model, OCP, and solver.
2. Run `run_controller_simulation(...)` and inspect the plots.
3. Display the short simulated video with `animate_swingup_simulation(...)`.
4. Only after the simulated motion looks plausible, rebuild the solver and
   try the commented `run_controller(...)` call on the real system.

The required experiments are:

1. **Fully actuated double pendulum** — both joints actuated.
2. **Pendubot** — only **joint 1** is actuated, joint 2 is almost free.

The **acrobot** is a bonus/challenge experiment. It is much harder on this
small, frictional robot because joint 1 can only be moved indirectly through
joint 2. Treat it as a discussion case about feasibility, local minima, and
the limits of plain tracking NMPC rather than as a required success case.


### 5.1 Fully actuated swing-up

Pick the parameters, build model → OCP → solver, simulate the controller,
and only then try the same setup on the real system.


In [ ]:
# --- tuning parameters for the FULLY ACTUATED case ---
pred_horizon = 20
Tf_mpc       = 0.5
mpc_dt       = Tf_mpc / pred_horizon
control_dt   = 0.001

weights = {
    "q":    [1000.0, 1000.0],
    "dq":   [ 100.0,  100.0],
    "u":    [   200.,    200.],
    "q_e":  [1_00_000.0, 1_00_000.0],
    "dq_e": [     10000.0,     10000.0],
}

v_max   = 10.0
u_max_1 = 0.14        # fully actuated — both motors used
u_max_2 = 0.14

model  = build_model()
ocp    = get_ocp(model, pred_horizon, mpc_dt, control_dt, weights,
                 v_max, u_max_1, u_max_2)
solver = get_solver(ocp)

# First test the exact same NMPC in simulation.
result_fa_sim = run_controller_simulation(solver, model,
                                          u_max_1=u_max_1, u_max_2=u_max_2,
                                          v_max=v_max,
                                          Tf_sim=5.0,
                                          sim_dt=0.002,
                                          mpc_period=0.02)
display(animate_swingup_simulation(result_fa_sim["ts"], result_fa_sim["xs"], skip=25))

# Once the simulated trajectory looks plausible, rebuild the solver and try hardware.
# solver = get_solver(ocp)
# result_fa = run_controller(solver,
#                            u_max_1=u_max_1, u_max_2=u_max_2, v_max=v_max,
#                            Tf_exp=5.0,
#                            experiment_type="DoublePendulum")


### 5.2 Pendubot (only joint 1 actuated)

Keep everything identical except the torque limits. Simulate first; if the
solver has trouble, allow a tiny residual torque on the unactuated joint 2
or increase the prediction horizon.


In [ ]:
# Pendubot: joint 1 actuated, joint 2 has a tiny residual torque.
u_max_1 = 0.14
u_max_2 = 0.02           # tiny residual torque on the unactuated joint

# You may also need a longer prediction horizon for the underactuated case.
pred_horizon = 20
Tf_mpc       = 0.5
mpc_dt       = Tf_mpc / pred_horizon
control_dt   = 0.001

weights = {
    "q":    [1000.0, 1000.0],
    "dq":   [ 100.0,  100.0],
    "u":    [   250.,    250.],
    "q_e":  [1_00_000.0, 1_00_000.0],
    "dq_e": [     10000.0,     10000.0],
}

v_max   = 10.0

model  = build_model()
ocp    = get_ocp(model, pred_horizon, mpc_dt, control_dt, weights,
                 v_max, u_max_1, u_max_2)
solver = get_solver(ocp)

# First test the exact same NMPC in simulation.
result_pendubot_sim = run_controller_simulation(solver, model,
                                                u_max_1=u_max_1, u_max_2=u_max_2,
                                                v_max=v_max,
                                                Tf_sim=5.0,
                                                sim_dt=0.002,
                                                mpc_period=0.02)
display(animate_swingup_simulation(result_pendubot_sim["ts"], result_pendubot_sim["xs"], skip=25))

# Once the simulated trajectory looks plausible, rebuild the solver and try hardware.
# solver = get_solver(ocp)
# result_pendubot = run_controller(solver,
#                                  u_max_1=u_max_1, u_max_2=u_max_2, v_max=v_max,
#                                  Tf_exp=5.0,
#                                  experiment_type="Pendubot")


### 5.3 Bonus: Acrobot (only joint 2 actuated)

This is a bonus/challenge experiment. Simulate it and inspect what happens,
but do not assume it will reliably swing up on the real CloudPendulum.

A common outcome is that the controller moves/folds the elbow (`q2`) but the
base link (`q1`) does not build enough energy to swing up; it illustrates a real limitation of local tracking NMPC
for highly underactuated swing-up. The optimizer may find a cheap local
compromise instead of the nonlocal energy-pumping maneuver needed for an
acrobot.


In [ ]:
# Acrobot: joint 1 (almost) free, joint 2 actuated.
u_max_1 = 0.02          # tiny residual torque for numerical robustness
u_max_2 = 0.14

pred_horizon = 40
Tf_mpc       = 2.0
mpc_dt       = Tf_mpc / pred_horizon
control_dt   = 0.02

weights = {
    "q":    [ 150.0,  80.0],
    "dq":   [   5.0,   5.0],
    "u":    [  20.0,   2.0],
    "q_e":  [5000.0, 2000.0],
    "dq_e": [ 200.0,  200.0],
}

v_max = 20.0

model  = build_model()
ocp    = get_ocp(model, pred_horizon, mpc_dt, control_dt, weights,
                 v_max, u_max_1, u_max_2)
solver = get_solver(ocp)

# First test the exact same NMPC in simulation.
# For acrobot, a typical result is an elbow-folding local optimum rather than
# a full swing-up. That is the phenomenon discussed in the markdown above.
result_acrobot_sim = run_controller_simulation(solver, model,
                                               u_max_1=u_max_1, u_max_2=u_max_2,
                                               v_max=v_max,
                                               Tf_sim=8.0,
                                               sim_dt=0.002,
                                               mpc_period=control_dt,
                                               motor_response=1.0)
display(animate_swingup_simulation(result_acrobot_sim["ts"], result_acrobot_sim["xs"], skip=25))

# Bonus only: try hardware manually if the simulation looks convincing.
# solver = get_solver(ocp)
# result_acrobot = run_controller(solver,
#                                 u_max_1=u_max_1, u_max_2=u_max_2, v_max=v_max,
#                                 Tf_exp=8.0,
#                                 experiment_type="Acrobot")
